In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import os

seedlist = [42, 834, 692, 489, 901, 408, 819, 808, 531, 166]

seed = seedlist[0]
os.environ['PYTHONHASHSEED'] = str(seed)

In [3]:
import pandas as pd
import numpy as np
import pickle
import re
import torch
import random
import math
import json

from pathlib import Path
from functools import lru_cache
from bisect import bisect_right
from tqdm.auto import tqdm

from clang import cindex
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaConfig, RobertaForMaskedLM, RobertaForSequenceClassification, Trainer, TrainingArguments

from tokenizers import Tokenizer, NormalizedString, PreTokenizedString, normalizers, decoders, processors, pre_tokenizers
from tokenizers.normalizers import StripAccents, unicode_normalizer_from_str, Replace
from tokenizers.processors import TemplateProcessing
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import PreTokenizer
from typing import Any, Callable, Dict, List, NewType, Optional, Tuple, Union
from transformers.tokenization_utils_base import BatchEncoding
from torch.nn.utils.rnn import pad_sequence
from transformers.trainer_utils import get_last_checkpoint

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


In [4]:
def _collate_batch(examples, tokenizer, pad_to_multiple_of: Optional[int] = None):

    tokenizer_pad_size = 'right'
    tokenizer_pad_id = 1

    """Collate `examples` into a batch, using the information in `tokenizer` for padding if necessary."""
    # Tensorize if necessary.
    if isinstance(examples[0], (list, tuple)):
        examples = [torch.tensor(e, dtype=torch.long) for e in examples]

    # Check if padding is necessary.
    length_of_first = examples[0].size(0)
    are_tensors_same_length = all(x.size(0) == length_of_first for x in examples)
    if are_tensors_same_length and (pad_to_multiple_of is None or length_of_first % pad_to_multiple_of == 0):
        return torch.stack(examples, dim=0)

    # If yes, check if we have a `pad_token`.
#     if tokenizer._pad_token is None:
#         raise ValueError(
#             "You are attempting to pad samples but the tokenizer you are using"
#             f" ({tokenizer.__class__.__name__}) does not have a pad token."
#         )

    # Creating the full tensor and filling it with our data.
    max_length = max(x.size(0) for x in examples)
    if pad_to_multiple_of is not None and (max_length % pad_to_multiple_of != 0):
        max_length = ((max_length // pad_to_multiple_of) + 1) * pad_to_multiple_of
    result = examples[0].new_full([len(examples), max_length], tokenizer_pad_id)
    for i, example in enumerate(examples):
        if tokenizer_pad_size == "right":
            result[i, : example.shape[0]] = example
        else:
            result[i, -example.shape[0] :] = example
    return result

In [5]:
def get_special_tokens_mask( token_ids_0: List, token_ids_1: Optional[List] = None, already_has_special_tokens: bool = False
) -> List[int]:

    ## The ids of special tokens from tokenizer
    all_special_ids = [0,1,2,3,4]

    special_tokens_mask = [1 if token in all_special_ids else 0 for token in token_ids_0]

    return special_tokens_mask

In [6]:
class CustomDataCollatorForLanguageModeling:

    tokenizer: Tokenizer
    mlm: bool = True
    mlm_probability: float = 0.15
    pad_to_multiple_of: Optional[int] = None
    tokenizer_mask_id = 4
    tokenizer_pad_id = 1

    def __init__(self,tokenizer,mlm,mlm_probability):
        self.tokenizer = tokenizer
        self.mlm = mlm
        self.mlm_probability = mlm_probability

    def __post_init__(self):
        if self.mlm and self.tokenizer.mask_token is None:
            raise ValueError(
                "This tokenizer does not have a mask token which is necessary for masked language modeling. "
                "You should pass `mlm=False` to train on causal language modeling instead."
            )

    def __call__(self, examples: List[Union[List[int], torch.Tensor, Dict[str, torch.Tensor]]]) -> Dict[str, torch.Tensor]:

        # Handle dict or lists with proper padding and conversion to tensor.
        if isinstance(examples[0], (dict, BatchEncoding)):
            batch = self.tokenizer.pad(examples, return_tensors="pt", pad_to_multiple_of=self.pad_to_multiple_of)
        else:
            batch = {"input_ids": _collate_batch(examples, self.tokenizer, pad_to_multiple_of=self.pad_to_multiple_of)}

        # If special token mask has been preprocessed, pop it from the dict.
        special_tokens_mask = batch.pop("special_tokens_mask", None)

        if self.mlm:
            batch["input_ids"], batch["labels"] = self.mask_tokens(
                batch["input_ids"], special_tokens_mask=special_tokens_mask
            )
        else:
            labels = batch["input_ids"].clone()
            if self.tokenizer_pad_id is not None:
                labels[labels == self.tokenizer_pad_id] = -100
            batch["labels"] = labels
        return batch

    def mask_tokens(self, inputs: torch.Tensor, special_tokens_mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Prepare masked tokens inputs/labels for masked language modeling: 80% MASK, 10% random, 10% original.
        """
        labels = inputs.clone()
        # We sample a few tokens in each sequence for MLM training (with probability `self.mlm_probability`)
        probability_matrix = torch.full(labels.shape, self.mlm_probability)
        if special_tokens_mask is None:
            special_tokens_mask = [
                get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
            ]
            special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        else:
            special_tokens_mask = special_tokens_mask.bool()

        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
        masked_indices = torch.bernoulli(probability_matrix).bool()
        labels[~masked_indices] = -100  # We only compute loss on masked tokens

        # 80% of the time, we replace masked input tokens with tokenizer.mask_token ([MASK])
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
        inputs[indices_replaced] = self.tokenizer_mask_id

        # 10% of the time, we replace masked input tokens with random word
        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
        random_words = torch.randint(self.tokenizer.get_vocab_size(), labels.shape, dtype=torch.long)
        inputs[indices_random] = random_words[indices_random]

        # The rest of the time (10% of the time) we keep the masked input tokens unchanged
        return inputs, labels

In [7]:
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.enabled = True
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
## Tokenizer
class MyTokenizer:

    cidx = cindex.Index.create()


    def clang_split(self, i: int, normalized_string: NormalizedString) -> List[NormalizedString]:
        ## Tokenize using clang
        tok = []
        tu = self.cidx.parse('tmp.c',
                       args=[''],
                       unsaved_files=[('tmp.c', str(normalized_string.original))],
                       options=0)
        for t in tu.get_tokens(extent=tu.cursor.extent):
            spelling = t.spelling.strip()

            if spelling == '':
                continue

            #spelling = spelling.replace(' ', '')
            tok.append(NormalizedString(spelling))

        return(tok)

    def pre_tokenize(self, pretok: PreTokenizedString):
        pretok.split(self.clang_split)

In [9]:
## Load pre-trained tokenizers
vocab_path = "/content/drive/MyDrive/VulnSight/tokenizer/drapgh-vocab.json"
merge_path = "/content/drive/MyDrive/VulnSight/tokenizer/drapgh-merges.txt"

vocab, merges = BPE.read_file(vocab=vocab_path, merges=merge_path)

my_tokenizer = Tokenizer(BPE(vocab, merges, unk_token="<unk>"))

my_tokenizer.normalizer = normalizers.Sequence([StripAccents(), Replace(" ", "Ä")])
my_tokenizer.pre_tokenizer = PreTokenizer.custom(MyTokenizer())

my_tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    special_tokens=[
    ("<s>",0),
    ("<pad>",1),
    ("</s>",2),
    ("<unk>",3),
    ("<mask>",4)
    ]
)

In [ ]:
# ============================================================
# loads tokenized chunks from Google Drive
# ============================================================

class PretrainDataset(Dataset):
    def __init__(self, cache_dir):
        self.cache_dir = Path(cache_dir)
        self.chunk_paths = sorted(self.cache_dir.glob("chunk_*.pkl"))

        if not self.chunk_paths:
            raise RuntimeError(
                f"No tokenized chunks found in {self.cache_dir}. "
                "Run the preprocessing cell first."
            )

        self.chunk_lengths = []
        self.cumulative_lengths = []
        total = 0

        for path in tqdm(self.chunk_paths, desc="Indexing cache", unit="chunk"):
            with open(path, "rb") as f:
                examples = pickle.load(f)

            if not isinstance(examples, list) or len(examples) == 0:
                raise RuntimeError(f"Invalid or empty tokenized chunk: {path}")

            self.chunk_lengths.append(len(examples))
            total += len(examples)
            self.cumulative_lengths.append(total)

        self.total_examples = total

    @lru_cache(maxsize=8)
    def _load_chunk(self, chunk_idx):
        # Keep only a few chunks in RAM; the full tokenized dataset stays on Drive.
        with open(self.chunk_paths[chunk_idx], "rb") as f:
            return pickle.load(f)

    def __len__(self):
        return self.total_examples

    def __getitem__(self, idx):
        if idx < 0:
            idx += self.total_examples
        if idx < 0 or idx >= self.total_examples:
            raise IndexError(idx)

        chunk_idx = bisect_right(self.cumulative_lengths, idx)
        previous_end = self.cumulative_lengths[chunk_idx - 1] if chunk_idx > 0 else 0
        local_idx = idx - previous_end

        example = self._load_chunk(chunk_idx)[local_idx]
        return torch.tensor(example, dtype=torch.long)


dataset = PretrainDataset(CACHE_DIR)
print(f"Tokenized training examples: {len(dataset):,}")
print("First example shape:", tuple(dataset[0].shape))


In [ ]:
DATA_PATH = "/content/drive/MyDrive/VulnSight/pretrain/drapgh.pkl"
CACHE_DIR = Path(
    "/content/drive/MyDrive/VulnSight/pretrain/tokenized_cache_v1"
)

MAX_LENGTH = 512
CHUNK_SIZE = 1000
PREPROCESS_SEED = 42

VOCAB_PATH = "/content/drive/MyDrive/VulnSight/tokenizer/drapgh-vocab.json"
MERGE_PATH = "/content/drive/MyDrive/VulnSight/tokenizer/drapgh-merges.txt"

CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cleaner(code):
    """Keep the original notebook's comment/newline/tab cleaning behavior."""
    if pd.isna(code):
        return ""

    code = str(code)
    pat = re.compile(r'(/\*([^*]|(\*+[^*/]))*\*+/)|(//.*)')
    code = re.sub(pat, '', code)
    code = re.sub('\n', '', code)
    code = re.sub('\t', '', code)
    return code


def file_signature(path):
    path = Path(path)
    stat = path.stat()
    return {
        "path": str(path),
        "size_bytes": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


# A manifest prevents accidentally mixing cached tokens from a different
# source file, vocabulary, merges, sequence length, or chunking configuration.
manifest = {
    "data": file_signature(DATA_PATH),
    "vocab": file_signature(VOCAB_PATH),
    "merges": file_signature(MERGE_PATH),
    "max_length": MAX_LENGTH,
    "chunk_size": CHUNK_SIZE,
    "shuffle_seed": PREPROCESS_SEED,
    "normalizer": 'Sequence(StripAccents(), Replace(" ", "Ä"))',
    "pretokenizer": "Clang MyTokenizer",
    "cleaner_version": "original-comment-newline-tab-cleaner-v1",
}

manifest_path = CACHE_DIR / "manifest.json"

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        old_manifest = json.load(f)
    if old_manifest != manifest:
        raise RuntimeError(
            f"Cache settings/source files do not match the existing cache at {CACHE_DIR}.\n"
            "To avoid mixing incompatible token IDs, change CACHE_DIR to a new folder "
            "(for example tokenized_cache_v2) and rerun this cell."
        )
else:
    existing_chunks = list(CACHE_DIR.glob("chunk_*.pkl"))
    if existing_chunks:
        raise RuntimeError(
            f"Found chunk files but no manifest in {CACHE_DIR}. "
            "For safety, change CACHE_DIR to a new folder and rerun."
        )
    temp_manifest = manifest_path.with_suffix(".tmp")
    with open(temp_manifest, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    os.replace(temp_manifest, manifest_path)

print("Loading pre-training dataset from Google Drive...")
mydata = pd.read_pickle(DATA_PATH)

if "functionSource" not in mydata.columns:
    raise ValueError("The pickle file must contain a 'functionSource' column.")

# Cleaning is repeated after a runtime reset; expensive tokenization is cached.
tqdm.pandas(desc="Cleaning source code")
codes = mydata["functionSource"].progress_apply(cleaner)
codes = codes.sample(frac=1, random_state=PREPROCESS_SEED).reset_index(drop=True)

total_examples = len(codes)
total_chunks = math.ceil(total_examples / CHUNK_SIZE)

print(f"Source-code functions: {total_examples:,}")
print(f"Tokenization chunks: {total_chunks:,}")
print(f"Cache directory: {CACHE_DIR}")

my_tokenizer.enable_truncation(max_length=MAX_LENGTH)

with tqdm(total=total_chunks, desc="Tokenizing chunks", unit="chunk") as progress:
    for chunk_idx in range(total_chunks):
        chunk_path = CACHE_DIR / f"chunk_{chunk_idx:06d}.pkl"

        # A saved chunk means this batch was already tokenized.
        if chunk_path.exists():
            progress.update(1)
            continue

        start = chunk_idx * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, total_examples)
        chunk_codes = codes.iloc[start:end].tolist()

        encoded = my_tokenizer.encode_batch(chunk_codes)
        examples = [item.ids for item in encoded]

        # Atomic rename helps avoid treating a partially written chunk as complete.
        temp_path = chunk_path.with_suffix(".tmp")
        with open(temp_path, "wb") as f:
            pickle.dump(examples, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(temp_path, chunk_path)

        progress.update(1)

print("Tokenization cache is ready.")


In [ ]:
model = RobertaForMaskedLM(config=config)
print(model.num_parameters())

my_tokenizer.enable_truncation(max_length=512)

data_collator = CustomDataCollatorForLanguageModeling(tokenizer=my_tokenizer,
                                                       mlm=True,
                                                       mlm_probability=0.15)

In [ ]:
                ## VulBERTa-small config

config = RobertaConfig(
        vocab_size=50000,
        max_position_embeddings=1026,
        num_attention_heads=3,
        num_hidden_layers=3,
        type_vocab_size=1,
)

                ## VulBERTa-medium config

# config = RobertaConfig(
#         vocab_size=50000,
#         max_position_embeddings=1026,
#         num_attention_heads=12,
#         num_hidden_layers=6,
#         type_vocab_size=1,
# )

                ## VulBERTa-base config

# config = RobertaConfig(
#         vocab_size=50000,
#         max_position_embeddings=1026,
#         num_attention_heads=12,
#         num_hidden_layers=12,
#         type_vocab_size=1,
# )

                ## VulBERTa-large config

# config = RobertaConfig(
#         vocab_size=50000,
#         max_position_embeddings=1026,
#         hidden_size =1024,
#         num_attention_heads=16,
#         num_hidden_layers=24,
#         intermediate_size=4096,
#         type_vocab_size=1,
# )

In [ ]:
# ============================================================
# VulSight: checkpointed and resumable MLM pre-training
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/VulnSight/models/"
    "VulSight_pretraining_small_v1"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    do_train=True,
    overwrite_output_dir=False,

    per_device_train_batch_size=8,
    num_train_epochs=7,

    # Save recovery checkpoints frequently to Google Drive.
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,

    # Show live progress and loss.
    logging_strategy="steps",
    logging_steps=50,
    disable_tqdm=False,

    seed=42,
    data_seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

# Resume from the latest checkpoint if one exists.
last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found. Starting pre-training from the beginning.")
    trainer.train()

# Save final model/training state and tokenizer after successful completion.
trainer.save_model(OUTPUT_DIR)
trainer.save_state()

TOKENIZER_SAVE_PATH = (
    "/content/drive/MyDrive/VulnSight/tokenizer/"
    "vulnsight-pretraining-tokenizer.json"
)
my_tokenizer.save(TOKENIZER_SAVE_PATH)

print("Pre-training run finished.")
print(f"Model saved to: {OUTPUT_DIR}")
print(f"Tokenizer saved to: {TOKENIZER_SAVE_PATH}")
